# 深度學習框架應用（大學部）｜主題 8：模型評估與視覺化

這本筆記本配合投影片《主題 8：模型評估與視覺化》，從一張混淆矩陣開始，一路做到調門檻、畫 PR／ROC 曲線、用 TensorBoard 盯訓練過程、挑出最佳 epoch，最後把分錯的圖叫出來做錯誤分析。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 20 片餅乾的混淆矩陣（TP／FP／FN／TN、P、R、F1、偷懶模型） | 第 115 頁 | 第 20～38 頁 |
| 二 | 調門檻、PR 曲線、ROC 與 AUC | 第 116 頁 | 第 50～60、102～103 頁 |
| 三 | 寫進 TensorBoard，找出最佳 epoch（Fashion-MNIST） | 第 117 頁 | 第 9～10、73～92 頁 |
| 四 | 多類別混淆矩陣與三種平均 | 第 118 頁 | 第 40～48、94～97 頁 |
| 五 | 錯題本：把最常搞混的圖叫出來 | 第 119 頁 | 第 93、98～99 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W08_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」，再由上往下做。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、ValueError、IndexError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 練習三～五用 Fashion-MNIST（第一次執行會自動下載，約 30 MB）。為了在 CPU 上幾分鐘就跑完，只抽 1000 張訓練、2000 張驗證、2000 張測試。有 GPU 會自動用，沒有也跑得完；在不同電腦上跑，練習三～五的數字會有一點點不同，這是正常的。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

五題都會用到下面這些套件，先跑這一格。TensorBoard 和 scikit-learn 在 Colab 上都已經裝好。

In [ ]:
# ===== 環境準備（直接執行）=====
import copy, datetime
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, classification_report,
                             precision_recall_curve, roc_curve, roc_auc_score)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch：', torch.__version__, '｜裝置：', dev)

## 練習一：20 片餅乾的混淆矩陣 (20%)

> 題目在投影片**第 115 頁**｜觀念在第 20～38 頁

**情境**：小智在點心工廠當品管員，一盤 20 片餅乾裡有 5 片烤焦（1）、15 片良品（0）。品管模型把其中 7 片判成烤焦：前 4 片烤焦的抓到了，第 5 片烤焦的漏掉了；另外還誤報了 3 片良品。

**這題在練**：用 `confusion_matrix` 數出四格，自己用四格算精確率 P、召回率 R、F1，再和 sklearn 的答案對一次；最後看一個「全部說良品」的偷懶模型，準確率多高、召回率多少

**步驟**

1. 在 `confusion_matrix(___, ___)` 填入答案和預測（順序：先答案、再預測），印出 2×2 的表；`cm.ravel()` 會依 sklearn 的排法拆成 TN、FP、FN、TP
2. 自己寫出精確率和召回率的分母，算出 P、R、F1，和下一行 sklearn 算的三個數字對答案
3. 看最後一行：偷懶模型的 accuracy 和 recall

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習一：20 片餅乾的混淆矩陣
y_true = np.array([1]*5 + [0]*15)      # 1＝烤焦、0＝良品
y_pred = np.array([1, 1, 1, 1, 0] + [1]*3 + [0]*12)
cm = confusion_matrix(___, ___)        # ✏️ 先答案、再預測
tn, fp, fn, tp = cm.ravel()            # sklearn 的排法
print(cm, '\nTP, FP, FN, TN =', tp, fp, fn, tn)
P = tp / (___)                         # ✏️ 精確率的分母
R = tp / (___)                         # ✏️ 召回率的分母
F1 = 2 * P * R / (P + R)
print(f'手算：P = {P:.4f}，R = {R:.4f}，F1 = {F1:.4f}')
print('sklearn F1 =', round(f1_score(y_true, y_pred), 4))
z = np.zeros(20, dtype=int)            # 偷懶：全部說良品
print('偷懶 accuracy =', accuracy_score(y_true, z))
print('偷懶 recall =', recall_score(y_true, z))

#### 📸 截圖 ①

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：混淆矩陣、TP／FP／FN／TN 那一行、手算和 sklearn 的 P、R、F1，以及偷懶模型那一行

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 從印出來的表讀出 TP、FP、FN、TN 各是多少。為什麼 sklearn 的表左上角是 TN，跟投影片上那張四格表的位置不一樣？**

（在這裡寫）

**2. 偷懶模型的 accuracy 有 0.75，為什麼還是沒用？在餅乾品管裡，FP（誤報）和 FN（漏抓）哪一個比較嚴重？說出理由。**

（在這裡寫）

## 練習二：調門檻、PR 曲線、ROC 與 AUC (20%)

> 題目在投影片**第 116 頁**｜觀念在第 50～60、102～103 頁

**情境**：模型替 12 片餅乾打了「烤焦分數」（越高越像烤焦）。前 5 片真的烤焦，後 7 片是良品。分數過了門檻就把餅乾攔下來。

**這題在練**：親手調門檻，看 P、R 怎麼一升一降；用 `precision_recall_curve` 找出 F1 最高的門檻；用 `roc_auc_score` 算 AUC（Area Under the Curve，曲線下面積）

**步驟**

1. 在迴圈裡填門檻變數：分數大於等於門檻就判成烤焦，印出門檻 0.3、0.5、0.7 的 P 和 R
2. `precision_recall_curve` 回傳的 `thr` 比 `prec`、`rec` 少一個，F1 要先去掉最後一個再找最大
3. 算 ROC-AUC，再執行下一格把 PR 曲線和 ROC（Receiver Operating Characteristic）曲線畫出來

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋、這一題印出的數字和兩張曲線圖
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習二：調門檻、PR 曲線、ROC 與 AUC
y = np.array([1]*5 + [0]*7)        # 5 片烤焦、7 片良品
s = np.array([.92, .80, .66, .55, .30,  # 烤焦的分數
              .70, .58, .40, .25, .15, .10, .05])
for t in [0.3, 0.5, 0.7]:
    pred = (s >= ___).astype(int)  # ✏️ 過門檻就攔下
    P, R = precision_score(y, pred), recall_score(y, pred)
    print(f'門檻 {t}：P = {P:.3f}，R = {R:.3f}')
prec, rec, thr = precision_recall_curve(y, s)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
i = f1[___].argmax()               # ✏️ 去掉最後一個
print(f'F1 最高：門檻 {thr[i]:.2f}，F1 = {f1[i]:.3f}')
auc = roc_auc_score(___, ___)      # ✏️ 先答案、再分數
print(f'ROC-AUC = {auc:.4f}')

In [ ]:
# 練習二：把兩條曲線畫出來（直接執行）
fpr, tpr, _ = roc_curve(y, s)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(rec, prec, 'o-'); ax[0].axhline(5 / 12, ls=':', color='gray')   # 基準線＝烤焦比例
ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].set_title('PR curve')
ax[1].plot(fpr, tpr, 'o-'); ax[1].plot([0, 1], [0, 1], ':', color='gray')  # 對角線＝亂猜
ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR'); ax[1].set_title(f'ROC curve, AUC = {auc:.3f}')
plt.tight_layout(); plt.show()

#### 📸 截圖 ②

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三個門檻的 P、R，F1 最高的門檻，ROC-AUC，以及下一格畫出的兩張曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 門檻從 0.5 調到 0.7，R 從 0.800 掉到 0.400，P 卻沒有升高（都是 0.667）。為什麼門檻調高，P 不一定變大？**

（在這裡寫）

**2. F1 最高的門檻是多少？如果漏抓烤焦餅乾的代價很高，你會把門檻設得比它高還是低？為什麼？**

（在這裡寫）

## 練習三：寫進 TensorBoard，找出最佳 epoch (20%)

> 題目在投影片**第 117 頁**｜觀念在第 9～10、73～92 頁

**情境**：用 Fashion-MNIST（10 類衣服和鞋子的灰階小圖，28×28）訓練一個全連接網路。故意只給 1000 張訓練圖，讓它練 20 個 epoch，看得到「練過頭」：訓練 loss 一路降，驗證 loss 降到某個 epoch 之後開始回升。

| 編號 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| 英文（圖上用） | T-shirt | Trouser | Pullover | Dress | Coat | Sandal | Shirt | Sneaker | Bag | Boot |
| 中文 | T 恤 | 褲子 | 套頭毛衣 | 洋裝 | 外套 | 涼鞋 | 襯衫 | 運動鞋 | 包包 | 短靴 |

**這題在練**：用 `SummaryWriter` 把每個 epoch 的 train loss、val loss、val acc 寫進 TensorBoard；用驗證 loss 挑出最佳 epoch，並把那一版的權重存下來（練習四要用）

**步驟**

1. 先執行「準備資料與模型」那一格（會下載 Fashion-MNIST、抽樣、建模型、定義 `run_epoch`）
2. 在 ✏️ 格填好 `add_scalar` 的數值和步數、比較最低 loss 的變數、最後關掉 writer，練 20 個 epoch
3. 執行後面兩格：一格用 matplotlib 畫出兩條 loss 曲線並印出最佳 epoch，一格在 Colab 裡打開 TensorBoard

> 想重跑 ✏️ 格，要先重跑「準備資料與模型」，模型才會回到還沒學過的起點。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋、印出的最佳 epoch 那兩行和 loss 曲線圖（TensorBoard 的 SCALARS 畫面也拍進來更好）
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備資料與模型（直接執行，不用改）=====
# 想重跑下面的 ✏️ 格，要先重跑這一格，模型才會回到還沒學過的起點
tf = transforms.ToTensor()
full = datasets.FashionMNIST('data', train=True, download=True, transform=tf)
test = datasets.FashionMNIST('data', train=False, download=True, transform=tf)
names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
         'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']          # 0～9 類（畫圖用英文）
g = torch.Generator().manual_seed(42)                          # 固定抽樣，大家抽到同一批
idx = torch.randperm(len(full), generator=g).tolist()
train_ds = Subset(full, idx[:1000])                            # 訓練：1000 張（故意少，看得到過擬合）
val_ds = Subset(full, idx[1000:3000])                          # 驗證：2000 張，拿來挑最佳 epoch
test_ds = Subset(test, torch.randperm(len(test), generator=g)[:2000].tolist())  # 測試：2000 張，最後才用
torch.manual_seed(0)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256)
test_dl = DataLoader(test_ds, batch_size=256)                  # 不打亂，順序固定
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(),
                      nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10)).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
EPOCHS, hist = 20, []                                          # hist 記每個 epoch 的 (train loss, val loss, val acc)
stamp = datetime.datetime.now().strftime('%m%d_%H%M')          # 資料夾名稱帶時間，例如 1014_1430

def run_epoch(dl, train):
    """跑過一個 DataLoader；train=True 會更新參數。回傳 (平均 loss, accuracy)。"""
    model.train(train)
    total = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(dev), y.to(dev)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(y); n += len(y)
            correct += (out.argmax(1) == y).sum().item()
    return total / n, correct / n

print('訓練／驗證／測試：', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# ✏️ 練習三：寫進 TensorBoard，找出最佳 epoch
writer = SummaryWriter(f'runs/mlp_lr1e-3_{stamp}')
best = float('inf')     # 目前最低的 val loss
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl, train=False)
    writer.add_scalar('loss/train', ___, ep)  # ✏️
    writer.add_scalar('loss/val', ___, ep)  # ✏️
    writer.add_scalar('acc/val', va_acc, ___)  # ✏️ 步數
    hist.append((tr_loss, va_loss, va_acc))
    if va_loss < ___:   # ✏️ 創新低？
        best, best_ep = va_loss, ep
        best_state = copy.deepcopy(model.state_dict())
writer.___()            # ✏️ 寫完要關

In [ ]:
# 練習三：看訓練曲線（直接執行）
tl, vl, va = zip(*hist)
print(f'最佳 epoch = {best_ep}，val loss = {best:.4f}，val acc = {va[best_ep - 1]:.3f}')
print(f'最後 epoch = {EPOCHS}，val loss = {vl[-1]:.4f}，train loss = {tl[-1]:.4f}')
ep_axis = range(1, EPOCHS + 1)
plt.figure(figsize=(7, 4))
plt.plot(ep_axis, tl, 'o-', label='train loss'); plt.plot(ep_axis, vl, 's-', label='val loss')
plt.axvline(best_ep, ls='--', color='gray', label=f'best epoch = {best_ep}')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.show()

In [ ]:
# 在 Colab 裡打開 TensorBoard（直接執行）：SCALARS 分頁看 loss、acc 三條曲線
%load_ext tensorboard
%tensorboard --logdir runs

#### 📸 截圖 ③

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：「最佳 epoch」「最後 epoch」兩行和 loss 曲線圖（有 TensorBoard 的 SCALARS 畫面更好）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 你的最佳 epoch 是第幾個？在它之後，train loss 和 val loss 各怎麼走？這種現象叫什麼？**

（在這裡寫）

**2. 為什麼用驗證集挑最佳 epoch，不用測試集挑？**

（在這裡寫）

## 練習四：多類別混淆矩陣與三種平均 (20%)

> 題目在投影片**第 118 頁**｜觀念在第 40～48、94～97 頁

**情境**：把練習三驗證 loss 最低的那一版權重換回來，對 2000 張測試圖跑一次（這是測試集唯一的一次）。10 類各約 200 張，看看哪一類最弱、哪兩類最常搞混。

**這題在練**：多類別的混淆矩陣、逐列正規化（`keepdims=True`）、macro／weighted 兩種平均，以及用「把對角線清成 0」找出最常搞混的一格

**步驟**

1. 先執行「用最佳 epoch 的模型跑一次測試集」那一格，得到 `ys`（答案）、`ps`（預測）、`probs`（每一類的機率）
2. 在 ✏️ 格填好：混淆矩陣的兩個參數、`keepdims`、macro 平均的名字、對角線要清成多少
3. 執行下一格：印出逐類別報告、畫出逐列正規化的混淆矩陣，並把圖送進 TensorBoard 的 IMAGES 分頁

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋、✏️ 格印出的幾行和混淆矩陣圖
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：用最佳 epoch 的模型跑一次測試集（直接執行）=====
model.load_state_dict(best_state)          # 換回驗證 loss 最低的那一版
model.eval()
ys, ps, probs = [], [], []                 # 答案、預測、每一類的機率
with torch.no_grad():
    for x, y in test_dl:                    # test_dl 沒有打亂，順序固定
        out = model(x.to(dev))
        probs.append(F.softmax(out, 1).cpu())
        ps += out.argmax(1).cpu().tolist()
        ys += y.tolist()
probs = torch.cat(probs).numpy()
print('測試集張數：', len(ys), '｜機率表形狀：', probs.shape)

In [ ]:
# ✏️ 練習四：多類別混淆矩陣與三種平均
cm = confusion_matrix(___, ___)    # ✏️ 先答案、再預測
cmn = cm / cm.sum(axis=1, keepdims=___)  # ✏️ 逐列
print('每一列加起來：', cmn.sum(axis=1).round(3))
print('accuracy =', round(accuracy_score(ys, ps), 3))
for avg in [___, 'weighted']:      # ✏️ 每類一票
    f1 = f1_score(ys, ps, average=avg)
    print(avg, 'F1 =', round(f1, 3))
off = cm.copy()
np.fill_diagonal(off, ___)         # ✏️ 對角線清成 0
i, j = np.unravel_index(off.argmax(), off.shape)
print(f'最常搞混：{names[i]}→{names[j]}，{cm[i, j]} 張')

In [ ]:
# 練習四：逐類別報告＋混淆矩陣圖，也送進 TensorBoard 的 IMAGES 分頁（直接執行）
print(classification_report(ys, ps, target_names=names, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cmn, display_labels=names).plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
plt.xticks(rotation=45)
writer = SummaryWriter(f'runs/mlp_lr1e-3_{stamp}')
writer.add_figure('confusion_matrix', fig, global_step=best_ep, close=False)
writer.close()
plt.show()

#### 📸 截圖 ④

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：每一列加起來、accuracy、macro／weighted F1、最常搞混的那一行，以及逐列正規化的混淆矩陣圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. accuracy、macro-F1、weighted-F1 三個數字，哪兩個比較接近？為什麼？**

（在這裡寫）

**2. 最常搞混的是哪兩類？在逐列正規化的圖上，那一格的比例是多少？它代表什麼意思？**

（在這裡寫）

## 練習五：錯題本：把最常搞混的圖叫出來 (20%)

> 題目在投影片**第 119 頁**｜觀念在第 93、98～99 頁

**情境**：練習四找到最常搞混的一格（真的是第 i 類、被猜成第 j 類）。這題把那一格裡的圖叫出來，依「模型有多確定自己是對的」排序，最離譜的排在前面，像在整理錯題本。

**這題在練**：用布林遮罩挑出「真 i、猜 j」的樣本，用 `np.argsort` 依信心由大到小排序，把前 8 張畫出來，看懂模型錯在哪裡

**步驟**

1. 先算整個測試集一共錯幾張（答案和預測不一樣的張數）
2. 用 `(ya == ___) & (pa == ___)` 挑出那一格的樣本；`np.argsort` 預設由小排到大，要由大到小就在前面加負號
3. 看畫出來的 8 張圖，每張上面的數字是模型猜成第 j 類的機率

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋、印出的兩行和 8 張圖
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習五：錯題本，把最常搞混的那一格叫出來
ya, pa = np.array(ys), np.array(ps)
print('測試集一共錯幾張：', (ya != ___).sum())  # ✏️
wrong = np.where((ya == ___) & (pa == ___))[0]  # ✏️ i, j
conf = probs[wrong, j]              # 猜成 j 的把握
top = wrong[np.argsort(___)][:8]    # ✏️ 由大到小
print(f'{names[i]} 被猜成 {names[j]}：{len(wrong)} 張')
fig, axes = plt.subplots(1, len(top), figsize=(12, 2.2))
for ax, k in zip(np.atleast_1d(axes), top):
    ax.imshow(test_ds[k][0].squeeze(), cmap='gray')
    ax.set_title(f'{probs[k, j]:.2f}'); ax.axis('off')
plt.show()

#### 📸 截圖 ⑤

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：「一共錯幾張」「被猜成…幾張」兩行，以及 8 張分錯的圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 看這幾張圖，你覺得模型為什麼會搞混？有沒有哪一張連你自己都分不出來，或是答案（標註）看起來就怪怪的？**

（在這裡寫）

**2. 提出兩個能減少這一格錯誤的做法。**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：precision_recall_curve 為什麼少一個門檻） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W08_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。